In [3]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn.utils.prune as prune

In [4]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

In [5]:

kmnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1918,), (0.3483,))  # KMNIST's own pixel mean/std
])

train_data_kmnist = torchvision.datasets.KMNIST('./data', train=True, transform=kmnist_transform, download=True)
test_data_kmnist  = torchvision.datasets.KMNIST('./data', train=False, transform=kmnist_transform, download=True)

train_loader_kmnist = DataLoader(train_data_kmnist, batch_size=64, shuffle=True)
test_loader_kmnist  = DataLoader(test_data_kmnist, batch_size=64, shuffle=False)

print(f"Train size: {len(train_data_kmnist)}")
print(f"Test size: {len(test_data_kmnist)}")

Train size: 60000
Test size: 10000


In [6]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.fc1 = nn.Linear(16*7*7, 64)
        self.fc2 = nn.Linear(64, 10)
        self.pool = nn.MaxPool2d(2)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)

Train the fresh model

In [ ]:

#model_kmnist = TinyNet()
#
#optimizer = torch.optim.Adam(model_kmnist.parameters(), lr=1e-3)
#
#criterion = nn.CrossEntropyLoss()

In [ ]:


#for epoch in range(5):
#    total_loss = 0
#    for images, labels in train_loader_kmnist:
#        optimizer.zero_grad()
#        output = model_kmnist(images)
#        loss = criterion(output, labels)
#        loss.backward()
#        optimizer.step()
#        total_loss += loss.item()
#    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader_kmnist):.4f}")
#
#torch.save(model_kmnist.state_dict(), 'tinynet_kmnist_baseline.pth')
#print("KMNIST baseline saved!")

Load the model

In [7]:
model_kmnist = TinyNet()
model_kmnist.load_state_dict(torch.load('tinynet_kmnist_baseline.pth'))
model_kmnist.eval()

TinyNet(
  (conv1): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=784, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (relu): ReLU()
)

In [8]:
def evaluate_kmnist(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in loader:
            output = model(images)
            _, predicted = torch.max(output, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / total
    print(f"KMNIST baseline accuracy: {acc:.2f}%")
    return acc

evaluate_kmnist(model_kmnist, test_loader_kmnist)

KMNIST baseline accuracy: 93.15%


93.15

In [9]:
def prune_model(model, amount):
    prune.l1_unstructured(model.conv1, name='weight', amount=amount)
    prune.l1_unstructured(model.conv2, name='weight', amount=amount)
    prune.l1_unstructured(model.fc1, name='weight', amount=amount)
    prune.l1_unstructured(model.fc2, name='weight', amount=amount)
    return model

def load_fresh_kmnist_model():
    model = TinyNet()
    model.load_state_dict(torch.load('tinynet_kmnist_baseline.pth'))
    model.eval()
    return model

for amount in [0.20, 0.40, 0.50, 0.60, 0.65, 0.70, 0.75, 0.80]:
    model = load_fresh_kmnist_model()
    model = prune_model(model, amount)
    print(f"Sparsity: {int(amount*100)}%", end=" | ")
    evaluate_kmnist(model, test_loader_kmnist)

Sparsity: 20% | KMNIST baseline accuracy: 93.21%
Sparsity: 40% | KMNIST baseline accuracy: 90.50%
Sparsity: 50% | KMNIST baseline accuracy: 85.33%
Sparsity: 60% | KMNIST baseline accuracy: 76.96%
Sparsity: 65% | KMNIST baseline accuracy: 72.25%
Sparsity: 70% | KMNIST baseline accuracy: 69.97%
Sparsity: 75% | KMNIST baseline accuracy: 51.11%
Sparsity: 80% | KMNIST baseline accuracy: 40.24%


In [10]:
checkpoints_kmnist = [0.20,0.21,0.22,0.23,0.24,0.25,0.26,0.27,0.28,0.29,0.30,0.31,0.32,0.33,0.34,0.35,0.36,0.37,0.38,0.39,0.40,0.41,0.42,0.43,0.44,0.45,0.46,0.47,0.48,0.49,0.50,
                       0.51,0.52,0.53,0.54,0.55,0.56,0.57,0.58,0.59,0.60,
                       0.62,0.64,0.66,0.68,0.70,0.72,0.74,0.76,0.78,0.80,0.81,
                       0.82,0.83,0.84,0.85,0.86,0.87,0.88,0.89,0.90]

accuracy_kmnist_full = []

for amount in checkpoints_kmnist:
    model = load_fresh_kmnist_model()
    model = prune_model(model, amount)
    print(f"Sparsity: {amount*100:.0f}%", end=" | ")
    acc = evaluate_kmnist(model, test_loader_kmnist)
    accuracy_kmnist_full.append(acc)

Sparsity: 20% | KMNIST baseline accuracy: 93.21%
Sparsity: 21% | KMNIST baseline accuracy: 93.06%
Sparsity: 22% | KMNIST baseline accuracy: 93.08%
Sparsity: 23% | KMNIST baseline accuracy: 93.16%
Sparsity: 24% | KMNIST baseline accuracy: 92.99%
Sparsity: 25% | KMNIST baseline accuracy: 93.10%
Sparsity: 26% | KMNIST baseline accuracy: 93.04%
Sparsity: 27% | KMNIST baseline accuracy: 93.01%
Sparsity: 28% | KMNIST baseline accuracy: 93.05%
Sparsity: 29% | KMNIST baseline accuracy: 92.84%
Sparsity: 30% | KMNIST baseline accuracy: 92.64%
Sparsity: 31% | KMNIST baseline accuracy: 92.42%
Sparsity: 32% | KMNIST baseline accuracy: 92.11%
Sparsity: 33% | KMNIST baseline accuracy: 91.99%
Sparsity: 34% | KMNIST baseline accuracy: 91.94%
Sparsity: 35% | KMNIST baseline accuracy: 92.07%
Sparsity: 36% | KMNIST baseline accuracy: 91.51%
Sparsity: 37% | KMNIST baseline accuracy: 91.22%
Sparsity: 38% | KMNIST baseline accuracy: 91.51%
Sparsity: 39% | KMNIST baseline accuracy: 90.88%
Sparsity: 40% | KMNI

In [11]:
import plotly.graph_objects as go

sparsity_pct_kmnist = [c * 100 for c in checkpoints_kmnist]

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sparsity_pct_kmnist,
    y=accuracy_kmnist_full,
    mode='lines+markers',
    line=dict(color='#7C3AED', width=2),
    marker=dict(size=6),
    hovertemplate='Sparsity: %{x:.0f}%<br>Accuracy: %{y:.2f}%<extra></extra>'
))
fig.update_xaxes(title_text='Sparsity (%)', dtick=5)
fig.update_yaxes(title_text='Accuracy (%)')
fig.update_layout(title='KMNIST — full pruning sweep (40–90%)', template='plotly_white')
fig.show()

SHOULDER CHECK AT 54-68% sparsity


In [15]:

shoulder_check = {}
for amount in [0.54,0.55,0.56,0.57, 0.58,0.59,0.60, 0.61,0.62,0.63,0.64,0.65,0.66,0.67, 0.68]:
    model = load_fresh_kmnist_model()
    model = prune_model(model, amount)          # all layers, your existing full-sweep prune
    acc = evaluate_kmnist(model, test_loader_kmnist)
    shoulder_check[amount] = acc
    print(f"{amount*100:.0f}% | {acc:.2f}%")

KMNIST baseline accuracy: 76.47%
54% | 76.47%
KMNIST baseline accuracy: 77.69%
55% | 77.69%
KMNIST baseline accuracy: 76.73%
56% | 76.73%
KMNIST baseline accuracy: 77.46%
57% | 77.46%
KMNIST baseline accuracy: 79.58%
58% | 79.58%
KMNIST baseline accuracy: 78.85%
59% | 78.85%
KMNIST baseline accuracy: 76.96%
60% | 76.96%
KMNIST baseline accuracy: 79.96%
61% | 79.96%
KMNIST baseline accuracy: 77.93%
62% | 77.93%
KMNIST baseline accuracy: 75.73%
63% | 75.73%
KMNIST baseline accuracy: 73.45%
64% | 73.45%
KMNIST baseline accuracy: 72.25%
65% | 72.25%
KMNIST baseline accuracy: 73.69%
66% | 73.69%
KMNIST baseline accuracy: 72.47%
67% | 72.47%
KMNIST baseline accuracy: 71.29%
68% | 71.29%


LAYER SENSITIVITY ANALYSIS

In [10]:
model_check = load_fresh_kmnist_model()
baseline_acc = evaluate_kmnist(model_check, test_loader_kmnist)

KMNIST baseline accuracy: 93.15%


In [12]:
layers = ['conv1','conv2','fc1','fc2']
c_points_layersensitivity = [0.40,0.60,0.70,0.74]
sensitivity = {}

In [ ]:


for val in c_points_layersensitivity:
    sensitivity[val] = {}
    print(f"\n=== Probe sparsity {val*100:.0f}% ===")
    for layer_name in layers:
        model = load_fresh_kmnist_model() #loading model new always beacuse one layer can cause the damage second time we dont need layer1+ layer 2 damage
        prune.l1_unstructured(getattr(model,layer_name),
                              name = 'weight', amount = val)
        acc  = evaluate_kmnist(model,test_loader_kmnist)
        drop = baseline_acc - acc
        sensitivity[val][layer_name] = drop
        print(f"  {layer_name:6s} | drop {drop:.2f}pp")
    ranked = sorted(sensitivity[val], key=sensitivity[val].get, reverse=True)

    print(f"  ranking (most sensitive first): {' > '.join(ranked)}")
   



=== Probe sparsity 40% ===
KMNIST baseline accuracy: 93.09%
  conv1  | drop 0.06pp
KMNIST baseline accuracy: 92.62%
  conv2  | drop 0.53pp
KMNIST baseline accuracy: 92.95%
  fc1    | drop 0.20pp
KMNIST baseline accuracy: 91.95%
  fc2    | drop 1.20pp
  ranking (most sensitive first): fc2 > conv2 > fc1 > conv1

=== Probe sparsity 60% ===
KMNIST baseline accuracy: 89.58%
  conv1  | drop 3.57pp
KMNIST baseline accuracy: 90.74%
  conv2  | drop 2.41pp
KMNIST baseline accuracy: 92.93%
  fc1    | drop 0.22pp
KMNIST baseline accuracy: 87.98%
  fc2    | drop 5.17pp
  ranking (most sensitive first): fc2 > conv1 > conv2 > fc1

=== Probe sparsity 70% ===
KMNIST baseline accuracy: 90.32%
  conv1  | drop 2.83pp
KMNIST baseline accuracy: 88.60%
  conv2  | drop 4.55pp
KMNIST baseline accuracy: 92.05%
  fc1    | drop 1.10pp
KMNIST baseline accuracy: 84.08%
  fc2    | drop 9.07pp
  ranking (most sensitive first): fc2 > conv2 > conv1 > fc1

=== Probe sparsity 74% ===
KMNIST baseline accuracy: 88.58%
  c

MORE FINER WEEP

In [17]:
layers = ['conv1','conv2','fc1','fc2']
c_points_layersensitivity = [0.40,0.42, 0.44, 0.46, 0.48, 0.50, 0.52, 0.54, 0.56, 0.58,0.60,0.61,0.62,0.63,0.64,0.65,0.66,0.67, 0.68,0.69,0.70,0.71,0.72,0.73,0.74,0.75,0.76,0.77,0.78,0.79,0.80,0.81,
                       0.82,0.83,0.84,0.85,0.86,0.87,0.88,0.89,0.90]
sensitivity = {}

for val in c_points_layersensitivity:
    sensitivity[val] = {}
    print(f"\n=== Probe sparsity {val*100:.0f}% ===")
    for layer_name in layers:
        model = load_fresh_kmnist_model() #loading model new always beacuse one layer can cause the damage second time we dont need layer1+ layer 2 damage
        prune.l1_unstructured(getattr(model,layer_name),
                              name = 'weight', amount = val)
        acc  = evaluate_kmnist(model,test_loader_kmnist)
        drop = baseline_acc - acc
        sensitivity[val][layer_name] = drop
        print(f"  {layer_name:6s} | drop {drop:.2f}pp")
    ranked = sorted(sensitivity[val], key=sensitivity[val].get, reverse=True)

    print(f"  ranking (most sensitive first): {' > '.join(ranked)}")
   



=== Probe sparsity 40% ===
KMNIST baseline accuracy: 93.09%
  conv1  | drop 0.06pp
KMNIST baseline accuracy: 92.62%
  conv2  | drop 0.53pp
KMNIST baseline accuracy: 92.95%
  fc1    | drop 0.20pp
KMNIST baseline accuracy: 91.95%
  fc2    | drop 1.20pp
  ranking (most sensitive first): fc2 > conv2 > fc1 > conv1

=== Probe sparsity 42% ===
KMNIST baseline accuracy: 92.98%
  conv1  | drop 0.17pp
KMNIST baseline accuracy: 92.44%
  conv2  | drop 0.71pp
KMNIST baseline accuracy: 93.03%
  fc1    | drop 0.12pp
KMNIST baseline accuracy: 91.65%
  fc2    | drop 1.50pp
  ranking (most sensitive first): fc2 > conv2 > conv1 > fc1

=== Probe sparsity 44% ===
KMNIST baseline accuracy: 92.69%
  conv1  | drop 0.46pp
KMNIST baseline accuracy: 92.53%
  conv2  | drop 0.62pp
KMNIST baseline accuracy: 93.06%
  fc1    | drop 0.09pp
KMNIST baseline accuracy: 91.55%
  fc2    | drop 1.60pp
  ranking (most sensitive first): fc2 > conv2 > conv1 > fc1

=== Probe sparsity 46% ===
KMNIST baseline accuracy: 92.44%
  c

In [20]:
import plotly.graph_objects as go

layers = ['conv1', 'conv2', 'fc1', 'fc2']

# rebuild x-axis and per-layer drop lists straight from your sweep results
probes_pct = [val * 100 for val in c_points_layersensitivity]

fine_sens = {layer: [sensitivity[val][layer] for val in c_points_layersensitivity]
             for layer in layers}

colors = {'conv1': '#E11D48',   # red
          'conv2': '#F59E0B',   # orange
          'fc1':   '#10B981',   # green
          'fc2':   '#7C3AED'}   # purple

fig = go.Figure()
for layer in layers:
    fig.add_trace(go.Scatter(
        x=probes_pct,
        y=fine_sens[layer],
        mode='lines+markers',
        name=layer,
        line=dict(color=colors[layer], width=2),
        marker=dict(size=5),
        hovertemplate=f'{layer}: %{{y:.2f}}pp at %{{x:.0f}}%<extra></extra>'
    ))

fig.update_xaxes(title_text='Sparsity of the pruned layer (%)', dtick=2)
fig.update_yaxes(title_text='Accuracy drop (pp)')
fig.update_layout(
    title='KMNIST — single-layer sensitivity, fine sweep (40–90%)',
    template='plotly_white',
    legend_title_text='Layer pruned'
)
fig.show()

In [19]:
import plotly.graph_objects as go

layers = ['conv1', 'conv2', 'fc1', 'fc2']

probes_pct = [val * 100 for val in c_points_layersensitivity]

# recover accuracy from stored drops: accuracy = baseline - drop
fine_acc = {layer: [baseline_acc - sensitivity[val][layer]
                    for val in c_points_layersensitivity]
            for layer in layers}

colors = {'conv1': '#E11D48',   # red
          'conv2': '#F59E0B',   # orange
          'fc1':   '#10B981',   # green
          'fc2':   '#7C3AED'}   # purple

fig = go.Figure()
for layer in layers:
    fig.add_trace(go.Scatter(
        x=probes_pct,
        y=fine_acc[layer],
        mode='lines+markers',
        name=layer,
        line=dict(color=colors[layer], width=2),
        marker=dict(size=5),
        hovertemplate=f'{layer}: %{{y:.2f}}%% at %{{x:.0f}}%%<extra></extra>'
    ))

# dashed grey anchor: the healthy model
fig.add_hline(y=baseline_acc, line_dash='dash', line_color='#9CA3AF',
              annotation_text=f'baseline {baseline_acc:.2f}%',
              annotation_position='top right')

fig.update_xaxes(title_text='Sparsity of the pruned layer (%)', dtick=2)
fig.update_yaxes(title_text='Accuracy (%)')
fig.update_layout(
    title='KMNIST — single-layer sensitivity, fine sweep (accuracy view)',
    template='plotly_white',
    legend_title_text='Layer pruned'
)
fig.show()

In [26]:
import plotly.graph_objects as go
all_points = sorted(sensitivity.keys())
probes_pct = [v * 100 for v in all_points]
gap = [max(sensitivity[v].values()) - min(sensitivity[v].values())
       for v in all_points]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=probes_pct, y=gap,
    mode='lines+markers',
    line=dict(color='#0EA5E9', width=2), marker=dict(size=6),
    name='sensitivity gap',
    hovertemplate='Gap: %{y:.2f}pp at %{x:.0f}%<extra></extra>'
))

# mark the full-network collapse zone for reference
fig.add_vrect(x0=70, x1=80, fillcolor='#F87171', opacity=0.12, line_width=0,
              annotation_text='full-network collapse zone', annotation_position='top left')

fig.update_xaxes(title_text='Sparsity (%)', dtick=2)
fig.update_yaxes(title_text='Sensitivity gap (pp)', dtick=2, range=[0, 40])
#fig.update_yaxes(title_text='Sensitivity gap (pp)')
fig.update_layout(title='KMNIST — sensitivity gap vs sparsity',
                  template='plotly_white')
fig.show()